In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.datasets import fashion_mnist


# =========================================================
# 1. 퍼셉트론 노트북
# =========================================================

def step_function(x):
    return 1 if x > 0 else 0


def perceptron(x1, x2, w1, w2, b):
    value = x1 * w1 + x2 * w2 + b
    return step_function(value)


def AND(x1, x2):
    return perceptron(x1, x2, 0.5, 0.5, -0.7)


def OR(x1, x2):
    return perceptron(x1, x2, 0.5, 0.5, -0.2)


def XOR(x1, x2):
    return 1 if AND(x1, x2) != OR(x1, x2) else 0


inputs = [(0, 0), (0, 1), (1, 0), (1, 1)]

result = []

for x1, x2 in inputs:
    result.append({
        "x1": x1,
        "x2": x2,
        "AND": AND(x1, x2),
        "OR": OR(x1, x2),
        "XOR": XOR(x1, x2)
    })

# 'perceptron()' 함수 + AND/OR/XOR 진리표 출력 포함
df_logic = pd.DataFrame(result)
display(df_logic)



# =========================================================
# 재현 — Fashion MNIST 단층 분류
# =========================================================

(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

# /255.0 정규화 적용
X = X_train_full / 255.0

# reshape(-1, 784) 적용
X = X.reshape(-1, 784)

# train_test_split(test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

model = Sequential([
    Input(shape=(784,)),

    # Dense(10, activation='softmax')
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',

    # sparse_categorical_crossentropy
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

# model.summary()
model.summary()

val_loss, val_accuracy = model.evaluate(X_val, y_val)
print("val_accuracy:", val_accuracy)

# val_accuracy ≥ 0.84 확인
print("val_accuracy >= 0.84:", val_accuracy >= 0.84)



# =========================================================
# Iris MLP — 단층 vs 2층 성능 비교
# =========================================================

iris = load_iris()
X = iris.data
y = iris.target

# StandardScaler 표준화 적용
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

single_model = Sequential([
    Input(shape=(4,)),

    # 단층: Dense(3, 'softmax')
    Dense(3, activation='softmax')
])

single_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# epochs=200으로 충분히 학습
single_history = single_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    verbose=0
)

single_loss, single_acc = single_model.evaluate(X_val, y_val, verbose=0)


two_layer_model = Sequential([
    Input(shape=(4,)),

    # 2층: Dense(8, 'sigmoid') → Dense(3, 'softmax')
    Dense(8, activation='sigmoid'),
    Dense(3, activation='softmax')
])

two_layer_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# epochs=200으로 충분히 학습
two_history = two_layer_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    verbose=0
)

two_loss, two_acc = two_layer_model.evaluate(X_val, y_val, verbose=0)

iris_result = pd.DataFrame({
    "모델": ["단층 MLP", "2층 MLP"],
    "val_accuracy": [single_acc, two_acc],
    "파라미터 수": [
        single_model.count_params(),
        two_layer_model.count_params()
    ]
})

display(iris_result)



# =========================================================
# Fashion MNIST 은닉층 구조 실험
# =========================================================

(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

X = X_train_full / 255.0
X = X.reshape(-1, 784)

# 동일한 train/val 데이터 사용 (재현 위해 동일 시드)
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)


def make_model(hidden_layers):
    model = Sequential()
    model.add(Input(shape=(784,)))

    for units in hidden_layers:
        model.add(Dense(units, activation='sigmoid'))

    model.add(Dense(10, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


experiments = {
    "단층": [],
    "64층": [64],
    "128층": [128],
    "64-32층": [64, 32]
}

experiment_result = []

for name, hidden_layers in experiments.items():
    model = make_model(hidden_layers)

    # epochs=10 동일하게 적용
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=32,
        verbose=0
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    experiment_result.append({
        "구조": name,
        "파라미터 수": model.count_params(),
        "val_accuracy": val_acc
    })

# 파라미터: 단층=7850, 64층=50890, 128층=101770, 64-32층=52650
df_experiment = pd.DataFrame(experiment_result)
display(df_experiment)



# =========================================================
# sklearn digits — ML vs DL 비교
# =========================================================

digits = load_digits()
X = digits.data
y = digits.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
logreg_acc = logreg.score(X_val, y_val)

dense_model = Sequential([
    Input(shape=(64,)),
    Dense(32, activation='sigmoid'),
    Dense(10, activation='softmax')
])

dense_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

dense_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    verbose=0
)

dense_loss, dense_acc = dense_model.evaluate(X_val, y_val, verbose=0)

digits_result = pd.DataFrame({
    "모델": ["LogisticRegression", "Dense 신경망"],
    "val_accuracy": [logreg_acc, dense_acc]
})

display(digits_result)

,x1,x2,AND,OR,XOR
0,0,0,0,0,0
1,0,1,0,1,1
2,1,0,0,1,1
3,1,1,1,1,0


Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7862 - loss: 0.6319 - val_accuracy: 0.8364 - val_loss: 0.4856
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 999us/step - accuracy: 0.8367 - loss: 0.4802 - val_accuracy: 0.8515 - val_loss: 0.4475
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 977us/step - accuracy: 0.8471 - loss: 0.4480 - val_accuracy: 0.8565 - val_loss: 0.4221
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 995us/step - accuracy: 0.8521 - loss: 0.4316 - val_accuracy: 0.8538 - val_loss: 0.4200
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 994us/step - accuracy: 0.8535 - loss: 0.4224 - val_accuracy: 0.8597 - val_loss: 0.4086
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8565 - loss: 0.4150 - val_accuracy: 0.8583 - val_loss: 0.4133
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8589 - loss: 0.4077 - val_accuracy: 0.8609 - val_loss: 0.4011
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1000us/step - accuracy: 0.8593 - loss

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 10)             │         7,850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,552 (92.00 KB)

...

 Optimizer params: 15,702 (61.34 KB)

375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 792us/step - accuracy: 0.8603 - loss: 0.4118
val_accuracy: 0.8603333234786987
val_accuracy >= 0.84: True


,모델,val_accuracy,파라미터 수
0,단층 MLP,0.733333,15
1,2층 MLP,0.833333,67


,구조,파라미터 수,val_accuracy
0,단층,7850,0.858833
1,64층,50890,0.884000
2,128층,101770,0.888417
3,64-32층,52650,0.881750


,모델,val_accuracy
0,LogisticRegression,0.972222
1,Dense 신경망,0.975000
